In [5]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
library(data.table)
library(ggplot2)
library(genoppi)

In [6]:
args <- list(
    in_vcf = "data/geneset/knockouts/ukb_eur_wes_200k_maf0to5e-2_pLoF_damaging_missense_combined.vcf.bgz",
    bridge = "/well/lindgren/flassen/ressources/genesets/genesets/data/biomart/220524_hgnc_ensg_enst_chr_pos.txt.gz"
)

In [47]:
command <- paste0('zcat ', args$in_vcf, '| grep -v "#" | cut -f1-3')
d <- fread(cmd=command)
n <- nrow(d)

In [48]:
colnames(d) <- c("chrom","id","ensembl_gene_id")

In [49]:
bridge <- fread(args$bridge)
bridge <- bridge[,c(1,2,5,6,7)]
bridge <- bridge[!duplicated(bridge)]

In [50]:
dt <- merge(d, bridge, all.x = TRUE)

In [51]:
data(msigdb_h_table)

In [52]:
geneset <- msigdb_h_table
colnames(geneset) <- c("hgnc_symbol","geneset")

In [54]:
mrg <- merge(dt, geneset, by = "hgnc_symbol", all.x = TRUE)
mrg <- mrg[!is.na(mrg$geneset),]
n_mrg <- nrow(mrg)

In [61]:
genesets <- unique(mrg$geneset)
out <- lapply(genesets, function(g){
  genes <- mrg$ensembl_gene_id[mrg$geneset %in% g]
  nas <- is.na(genes)
  genes <- genes[!nas]
  n <- length(genes)
  if (n > 0){
    row1 <- paste(c(g,'var', genes), collapse = " ")
    row2 <- paste(c(g, 'anno', rep(g, n)), collapse = " ")
    return(paste0(c(row1, row2), collapse = '\n'))
  }
})

print(out)

[[1]]
[1] "HALLMARK_ALLOGRAFT_REJECTION var ENSG00000090861 ENSG00000069399 ENSG00000012048 ENSG00000166278 ENSG00000105369 ENSG00000153283 ENSG00000115866 ENSG00000107581 ENSG00000117560 ENSG00000241106 ENSG00000204632 ENSG00000140749 ENSG00000122641 ENSG00000185507 ENSG00000113263 ENSG00000096968 ENSG00000043462 ENSG00000054219 ENSG00000100985 ENSG00000239672 ENSG00000081237 ENSG00000140986 ENSG00000113532 ENSG00000010327 ENSG00000168394 ENSG00000204267 ENSG00000174125 ENSG00000115085\nHALLMARK_ALLOGRAFT_REJECTION anno HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_ALLOGRAFT_REJECTION HALLMARK_

In [ ]:

M <- d[,c("csqs.gene_id", "varid", "consequence_category")]
colnames(M) <- c('gene','variant','anno')

genes <- unique(M$gene)


out <- null_omit(out)
writeLines(paste(out, collapse = '\n'), args$output_path)